In [4]:
import requests
import pandas as pd
import time
from datetime import datetime

# Credenciales (Twitch/IGDB)
CLIENT_ID = "i31ftaab0ooq0oa003t4tl8t3kop02"
ACCESS_TOKEN = "432d87rdz4h0d0hkt6v4apw9uh8606"

HEADERS = {
    "Client-ID": CLIENT_ID,
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

# Diccionario de categorias de IGDB
Categorias = {
    0: 'Juego Principal', 1: 'DLC', 2: 'Expansión', 3: 'Bundle',
    4: 'Expansión Independiente', 5: 'Mod', 6: 'Episodio', 7: 'Temporada',
    8: 'Remake', 9: 'Remaster', 10: 'Juego Expandido', 11: 'Port',
    12: 'Fork', 13: 'Pack', 14: 'Actualización'
}

def extraer_nombres(lista):
    """
    Función sencilla para sacar solo los nombres de las listas que devuelve la API.
    Si recibe una lista de diccionarios, devuelve un texto separado por comas.
    """
    if isinstance(lista, list):
        return ", ".join([item.get('name', '') for item in lista])
    return "N/A"

def extraer_datos_igdb(limite_juegos=30000):
    juegos = []

    print("Extracción de IGDB...")

    # Consultamos de 500 en 500 porque es el máximo permitido por la API
    for offset in range(0, limite_juegos, 500):

        # Consulta con las variables y el filtro de (>10 votos)
        query = f"""
            fields name, rating, rating_count, aggregated_rating, genres.name, platforms.name,
                   game_modes.name, player_perspectives.name, game_type, first_release_date;
            where rating != null & rating_count > 10;
            sort rating_count desc;
            limit 500;
            offset {offset};
        """

        respuesta = requests.post("https://api.igdb.com/v4/games", headers=HEADERS, data=query)

        if respuesta.status_code == 200:
            datos = respuesta.json()

            # Si la API devuelve una lista vacía, significa que ya no hay más juegos que cumplan el filtro
            if len(datos) == 0:
                break

            # Extraer y limpiar cada juego de la página actual
            for juego in datos:
                año = None
                if 'first_release_date' in juego:
                    año = datetime.fromtimestamp(juego['first_release_date']).year

                # Buscamos el valor real. Si no existe, "Desconocido"
                categoria_num = juego.get('game_type')

                if categoria_num is not None:
                    categoria_texto = Categorias.get(int(categoria_num), f'Desconocido ({categoria_num})')
                else:
                    # Si entra aquí, es que la API no devolvió el campo 'game_type'
                    categoria_texto = "No especificado"

                juego_limpio = {
                    'nombre': juego.get('name', 'N/A'),
                    'plataformas': extraer_nombres(juego.get('platforms')),
                    'año_lanzamiento': año,
                    'generos': extraer_nombres(juego.get('genres')),
                    'modos_juego': extraer_nombres(juego.get('game_modes')),
                    'perspectiva': extraer_nombres(juego.get('player_perspectives')),
                    'rating_usuarios': round(juego.get('rating', 0), 2),
                    'votos_usuarios': juego.get('rating_count', 0),
                    'rating_expertos': round(juego.get('aggregated_rating', 0), 2) if 'aggregated_rating' in juego else None,
                    'categoria_igdb': categoria_texto
                }
                juegos.append(juego_limpio)

            print(f"Descargados {len(juegos)} juegos...")
        else:
            print(f"Error en la consulta: {respuesta.status_code}")
            break

        time.sleep(0.5) # Pausa para no saturar el servidor

    # Convertir a df
    df = pd.DataFrame(juegos)

    # Ordenamos
    orden_columnas = [
        'nombre', 'plataformas', 'año_lanzamiento', 'generos', 'modos_juego',
        'perspectiva', 'rating_usuarios', 'votos_usuarios', 'rating_expertos', 'categoria_igdb'
    ]

    return df[orden_columnas]

df_igdb_final = extraer_datos_igdb(limite_juegos=30000)

# Guardar
df_igdb_final.to_csv("dataset_igdb.csv", index=False, encoding='utf-8')

print("Dimensiones de la tabla:", df_igdb_final.shape)

print(f"Total de registros obtenidos: {len(df_igdb_final)}")


Extracción de IGDB...
Descargados 500 juegos...
Descargados 1000 juegos...
Descargados 1500 juegos...
Descargados 2000 juegos...
Descargados 2500 juegos...
Descargados 3000 juegos...
Descargados 3500 juegos...
Descargados 4000 juegos...
Descargados 4500 juegos...
Descargados 5000 juegos...
Descargados 5500 juegos...
Descargados 6000 juegos...
Descargados 6500 juegos...
Descargados 7000 juegos...
Descargados 7500 juegos...
Descargados 8000 juegos...
Descargados 8500 juegos...
Descargados 9000 juegos...
Descargados 9500 juegos...
Descargados 9904 juegos...
Dimensiones de la tabla: (9904, 10)
Total de registros obtenidos: 9904
